In [1]:
%load_ext autoreload
%autoreload 2
from src.data_loading import load_data, PROCESSED_DATA_PATH
import pandas as pd
from sklearn.metrics import roc_auc_score
from src.evaluation import *
from src.training import run_lgbm_cv
import src.secondary_tables as st
application_train, application_test = load_data()
y = application_train[TARGET_COL]

auc_results = {}

def record_auc(stage, X, results):
    auc_results[stage] = {
        "num_features": X.shape[1],
        "mean_auc": results["mean_auc"],
        "std_auc": results["std_auc"],
        "oof_auc": roc_auc_score(y, results["oof_preds"]),
    }
    auc_table = pd.DataFrame.from_dict(auc_results, orient="index").reset_index()
    return auc_table.rename(columns={"index": "stage"})


In [2]:
from src.data_loading import ROOT, RAW_DATA_PATH
bureau = pd.read_csv(RAW_DATA_PATH / 'bureau.csv')
application_train_1, application_test_1 = st.add_bureau(application_train, application_test, bureau)
X_1 = application_train_1[[c for c in application_train_1.columns if c not in [TARGET_COL, ID_COL]]]
results = run_lgbm_cv(X_1, y)
summarize_cv_results(results, y)
record_auc("bureau", X_1, results)

mean_auc: 0.7691459777344394
std_auc: 0.0038654791966809023
oof_auc: 0.769134600103842


,stage,num_features,mean_auc,std_auc,oof_auc
0,bureau,197,0.769146,0.003865,0.769135


In [3]:
bureau = pd.read_csv(RAW_DATA_PATH / 'bureau.csv')
bureau = st.add_bureau_balance(bureau)
application_train_2, application_test_2 = st.add_bureau(application_train, application_test, bureau)
X_2 = application_train_2[[c for c in application_train_2.columns if c not in [TARGET_COL, ID_COL]]]
results = run_lgbm_cv(X_2, y)
summarize_cv_results(results, y)
application_train_2.to_csv(PROCESSED_DATA_PATH / 'application_train_2.csv')
application_test_2.to_csv(PROCESSED_DATA_PATH / 'application_test_2.csv')
record_auc("bureau + bureau_balance", X_2, results)

mean_auc: 0.7688440996459673
std_auc: 0.0029339156689773003
oof_auc: 0.7688293663207025


,stage,num_features,mean_auc,std_auc,oof_auc
0,bureau,197,0.769146,0.003865,0.769135
1,bureau + bureau_balance,227,0.768844,0.002934,0.768829


In [4]:
application_train_3, application_test_3 = st.add_previous_application(application_train_2, application_test_2)
X_3 = application_train_3[[c for c in application_train_3.columns if c not in [TARGET_COL, ID_COL]]]
results = run_lgbm_cv(X_3, y)
summarize_cv_results(results, y)
record_auc("bureau + bureau_balance + previous_application", X_3, results)

mean_auc: 0.7760739741832046
std_auc: 0.0031674065011340837
oof_auc: 0.7760486633477779


,stage,num_features,mean_auc,std_auc,oof_auc
0,bureau,197,0.769146,0.003865,0.769135
1,bureau + bureau_balance,227,0.768844,0.002934,0.768829
2,bureau + bureau_balance + previous_application,362,0.776074,0.003167,0.776049


In [5]:
application_train_4, application_test_4 = st.add_pos_cash_balance(application_train_3, application_test_3)
X_4 = application_train_4[[c for c in application_train_4.columns if c not in [TARGET_COL, ID_COL]]]
results = run_lgbm_cv(X_4, y)
summarize_cv_results(results, y)
record_auc("bureau + bureau_balance + previous_application + POS_CASH_balance", X_4, results)

mean_auc: 0.7782449357332052
std_auc: 0.0035834901230821824
oof_auc: 0.7782184656340733


,stage,num_features,mean_auc,std_auc,oof_auc
0,bureau,197,0.769146,0.003865,0.769135
1,bureau + bureau_balance,227,0.768844,0.002934,0.768829
2,bureau + bureau_balance + previous_application,362,0.776074,0.003167,0.776049
3,bureau + bureau_balance + previous_application...,399,0.778245,0.003583,0.778218


In [6]:
application_train_5, application_test_5 = st.add_credit_card_balance(application_train_4, application_test_4)
X_5 = application_train_5[[c for c in application_train_5.columns if c not in [TARGET_COL, ID_COL]]]
results = run_lgbm_cv(X_5, y)
summarize_cv_results(results, y)
record_auc("bureau + bureau_balance + previous_application + POS_CASH_balance + credit_card_balance", X_5, results)

mean_auc: 0.7802287992552941
std_auc: 0.003323393411916132
oof_auc: 0.7802134169712314


,stage,num_features,mean_auc,std_auc,oof_auc
0,bureau,197,0.769146,0.003865,0.769135
1,bureau + bureau_balance,227,0.768844,0.002934,0.768829
2,bureau + bureau_balance + previous_application,362,0.776074,0.003167,0.776049
3,bureau + bureau_balance + previous_application...,399,0.778245,0.003583,0.778218
4,bureau + bureau_balance + previous_application...,522,0.780229,0.003323,0.780213


In [7]:
application_train_6, application_test_6 = st.add_installments_payments(application_train_5, application_test_5)
application_train_6.to_csv(PROCESSED_DATA_PATH / 'application_train_with_secondary_tables.csv')
application_test_6.to_csv(PROCESSED_DATA_PATH / 'application_test_with_secondary_tables.csv')
X_6 = application_train_6[[c for c in application_train_6.columns if c not in [TARGET_COL, ID_COL]]]
results = run_lgbm_cv(X_6, y)
summarize_cv_results(results, y)
record_auc("all secondary tables", X_6, results)

mean_auc: 0.7833652138409148
std_auc: 0.00410877781829087
oof_auc: 0.7833413766611


,stage,num_features,mean_auc,std_auc,oof_auc
0,bureau,197,0.769146,0.003865,0.769135
1,bureau + bureau_balance,227,0.768844,0.002934,0.768829
2,bureau + bureau_balance + previous_application,362,0.776074,0.003167,0.776049
3,bureau + bureau_balance + previous_application...,399,0.778245,0.003583,0.778218
4,bureau + bureau_balance + previous_application...,522,0.780229,0.003323,0.780213
5,all secondary tables,568,0.783365,0.004109,0.783341
